# Stage 5 · Examples that resist learning (assignment 5.2)

**Question.** Which labelled objects do the trained models fail on again and again, are they one population or several, what is the likely reason for each, and can the failure be **predicted from the object's attributes** before looking at any result?

**Inputs: only what was trained on Kaggle.** The saved predictions (`<arm>/predictions/{dev,val}.csv`) of the final models `D1_s0` and `A1_s0` and of the D1 seed repeat `D1_s1` (a noise check on D1, not a third model). The subset arms of notebook 04 are not used. No training, no weights, no model of mine. The dataset is read for the ground truth and for the scene attributes.
The protocol, expectations and the decision rule below were written down before anything here was computed.

**What "resists learning" means here.** The Kaggle runs saved no predictions on the training images, so the objects analysed are the held-out ones: dev (66 images) and val (22 images). An object *resists* if **none of the three full-data models solves it**. The train-set version of the question (objects the detector cannot fit even when it has seen them) needs the D1 / A1 weights run over the training images and is **not done**.

| Signal | Definition (per model, at its own conf\*, chosen on dev and applied unchanged to val) |
|---|---|
| found | the highest-scoring detection at or above conf\* that overlaps the object at IoU >= 0.5 exists |
| solved | found, and that detection has the object's class |
| `R_miss` | solved by none of D1_s0, D1_s1, A1_s0 (both final models and the D1 seed repeat) |
| `R_notfound` | found by none of them (the class-agnostic version) |

**Integrity.** Dev defines every operating point and every fit; val is a descriptive replication and the test split of the predictor; nothing is selected on it.
**Declared rule.** "Attributes predict where the trained models fail" holds only if the **val** AUC for `R_miss` has its bootstrap lower bound above 0.5 **and** the top-fifth hit rate is at least 1.3 x the base rate.
**Not done:** train-set recovery, a blinded human re-audit of the populations (the check that separates ambiguity from a missing label), per-instance training dynamics, multi-label inference.

In [ ]:
import sys
from pathlib import Path

RUNTIME = "local"           # "local" | "kaggle"
if RUNTIME == "kaggle":
    SRC_DIR = Path("/kaggle/input/datasets/happyanurag/auric-src-subset")                 # the folder that CONTAINS auric/
    OUTPUTS = Path("/kaggle/input/datasets/happyanurag/auric-outputs/Auric-outputs")      # OUTPUTS/<arm>/predictions/{dev,val}.csv
    OUT_DIR = Path("/kaggle/working/resist")
else:
    ROOT = Path("..").resolve()                 # the submission folder (run the notebook from notebooks/)
    SRC_DIR, OUTPUTS = ROOT / "src", ROOT / "results" / "runs"            # OUTPUTS/<arm>/predictions/{dev,val}.csv[.gz]
    OUT_DIR = ROOT / "results" / "resist_learning"
OUT_DIR.mkdir(parents=True, exist_ok=True)
if str(SRC_DIR) in sys.path:
    sys.path.remove(str(SRC_DIR))
sys.path.insert(0, str(SRC_DIR))
CONFIG_FILE = SRC_DIR / "auric" / "configs" / "SC.yaml"               # data paths, split, size bins (the arms differ only in their training subset)
ENV_FILE = SRC_DIR / "auric" / "configs" / "env" / f"{RUNTIME}.yaml"

FULL = ["D1_s0", "D1_s1", "A1_s0"]                                   # the three full-data models that define R_miss
BOOTSTRAP, SEED, MIN_GROUP = 1000, 0, 20
print(f"Runtime: {RUNTIME} | package {SRC_DIR / 'auric'} | predictions {OUTPUTS}")

In [ ]:
import os
import time
from concurrent.futures import ThreadPoolExecutor

import cv2
import numpy as np
import pandas as pd

from auric.boxes import box_area, size_bucket_labels
from auric.config import load_config
from auric.data.dataset import index_split, load_annotations, read_class_names
from auric.data.splits import ensure_split
from auric.evaluation import evaluate
from auric.evaluation.classification import matched_pairs
from auric.structures import load_detections

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.precision", 3)

cfg = load_config(CONFIG_FILE, ENV_FILE)
DATA_ROOT, SPLIT_FILE, EC = Path(cfg["paths"]["data_root"]), Path(cfg["paths"]["split_file"]), cfg["evaluation"]
NAMES = read_class_names(DATA_ROOT)
SHORT = [n.replace("Truck", "").replace("w/", "").strip() for n in NAMES]
RECORDS = {s: index_split(DATA_ROOT, s) for s in ("train", "val")}
ANNS = {s: load_annotations(RECORDS[s], len(NAMES)) for s in RECORDS}
split, _ = ensure_split(SPLIT_FILE, ANNS["train"], NAMES, **cfg["split"])
DEV_IDS = set(split["dev"])
GT = {"dev": {i: ANNS["train"][i] for i in split["dev"]}, "val": {r.image_id: ANNS["val"][r.image_id] for r in RECORDS["val"]}}
REC = {"dev": {r.image_id: r for r in RECORDS["train"] if r.image_id in DEV_IDS}, "val": {r.image_id: r for r in RECORDS["val"]}}


def predictions_file(arm, part):
    # OUTPUTS/<arm>/predictions/<part>.csv (Kaggle dataset) or OUTPUTS/<run>/<run>_predictions/<part>.csv[.gz] (local, the first runs without the seed suffix)
    for name in (arm, arm[:-3]) if arm.endswith("_s0") else (arm,):
        for p in (OUTPUTS / name / "predictions" / f"{part}.csv", OUTPUTS / name / "predictions" / f"{part}.csv.gz",
                  OUTPUTS / name / f"{name}_predictions" / f"{part}.csv", OUTPUTS / name / f"{name}_predictions" / f"{part}.csv.gz"):
            if p.is_file():
                return p


AVAIL = [a for a in FULL if all(predictions_file(a, p) for p in ("dev", "val"))]
FULL_AVAIL = [a for a in FULL if a in AVAIL]
print("available:", AVAIL, "| missing:", [a for a in FULL if a not in AVAIL])
if len(FULL_AVAIL) < len(FULL):
    print("PROVISIONAL: the declared test needs all three full-data models.")
print({p: f"{len(GT[p])} images, {sum(len(a) for a in GT[p].values())} objects" for p in GT})

## Stage 1 · One row per labelled object, with its attributes

Dev and val objects. The attributes are read from the scene and the box only, nothing about any model's output: class, size (sqrt of the area; buckets from the evaluation config), crowding (other objects whose centre is within 2.5 box sides), border contact (box within 8 px of the scene edge), objects in the scene, scene brightness and sharpness (on a quarter-scale grey image).

In [ ]:
rows = []
for part in ("dev", "val"):
    for i, a in GT[part].items():
        for j in range(len(a)):
            rows.append((part, i, j, int(a.cls[j]), *a.boxes[j]))
T = pd.DataFrame(rows, columns=["part", "image", "j", "cls", "x1", "y1", "x2", "y2"])
START = T.reset_index().groupby(["part", "image"])["index"].min().to_dict()      # first row of each image: row = START[part, image] + j

box = T[["x1", "y1", "x2", "y2"]].to_numpy(float)
T["size"] = np.sqrt(box_area(box))
T["bucket"] = size_bucket_labels(T["size"].to_numpy(), EC["size_bins"])
T["n_scene"] = T.groupby(["part", "image"]).j.transform("size")
cx, cy = ((T.x1 + T.x2) / 2).to_numpy(), ((T.y1 + T.y2) / 2).to_numpy()
side = np.maximum(T.x2 - T.x1, T.y2 - T.y1).to_numpy()
crowd = np.zeros(len(T))
for _, g in T.groupby(["part", "image"]):
    i = g.index.to_numpy()
    d = np.hypot(cx[i][:, None] - cx[i][None], cy[i][:, None] - cy[i][None])
    crowd[i] = (d < 2.5 * side[i][:, None]).sum(1) - 1
T["crowd"] = crowd
W = np.array([REC[p][i].width for p, i in zip(T.part, T.image)]); H = np.array([REC[p][i].height for p, i in zip(T.part, T.image)])
T["border"] = (T.x1 < 8) | (T.y1 < 8) | (T.x2 > W - 8) | (T.y2 > H - 8)

def scene_stats(key):
    g = cv2.imread(str(REC[key[0]][key[1]].image_path), cv2.IMREAD_REDUCED_GRAYSCALE_4)       # full PNG decode: the slow part, and it releases the GIL
    return key, (float(g.mean()), float(cv2.Laplacian(g, cv2.CV_64F).var()))


t0 = time.perf_counter()
keys = list(T[["part", "image"]].drop_duplicates().itertuples(index=False, name=None))
with ThreadPoolExecutor(max_workers=min(os.cpu_count() or 1, 16)) as pool:
    stats = dict(pool.map(scene_stats, keys))
print(f"read {len(keys)} scenes with {pool._max_workers} threads in {time.perf_counter() - t0:.0f}s")
T["bright"] = [stats[p, i][0] for p, i in zip(T.part, T.image)]
T["sharp"] = [stats[p, i][1] for p, i in zip(T.part, T.image)]
T["class"] = [SHORT[k] for k in T.cls]
T["bright_q"] = pd.qcut(T.bright, 3, labels=["dark", "mid", "bright"])
T["scene_q"] = pd.qcut(T.n_scene, 3, labels=["sparse", "mid", "dense"], duplicates="drop")
T["crowded"] = T.crowd >= 3
display(T.groupby("part")[["size", "crowd", "n_scene", "bright", "sharp"]].describe().T.round(2))

## Stage 2 · Found and solved, per model

conf\* is the confidence that maximises the model's F1 on **dev** (`evaluate`); the same value is applied to val. Each object is paired with the highest-scoring detection at or above conf\* that overlaps it at IoU >= 0.5 (`matched_pairs`); a model that places a wrong-class box above a right-class box on the same object does not solve it.

In [ ]:
def outcomes(arm):
    conf, res = None, {}
    for part in ("dev", "val"):
        dets = load_detections(predictions_file(arm, part))
        if part == "dev":
            conf = evaluate(GT["dev"], dets, NAMES, EC["size_bins"], 0, SEED).summary["operating_point"]["conf"]
        pred = np.full(sum(len(a) for a in GT[part].values()), -1)
        base = T.index[T.part == part][0]
        for i, g in GT[part].items():
            d = dets.get(i)
            if d is None or not len(d) or not len(g):
                continue
            keep = d.scores >= conf
            pairs = matched_pairs(d.boxes[keep], d.scores[keep], g)
            if len(pairs):
                pred[START[part, i] - base + pairs[:, 1]] = d.cls[keep][pairs[:, 0]]
        res[part] = pred
    return conf, res


CONF, summary = {}, []
for a in AVAIL:
    CONF[a], res = outcomes(a)
    T[f"pred_{a}"] = np.concatenate([res["dev"], res["val"]])
    T[f"found_{a}"] = T[f"pred_{a}"] >= 0
    T[f"solved_{a}"] = T[f"pred_{a}"] == T.cls
    for part in ("dev", "val"):
        m = T.part == part
        summary.append({"arm": a, "split": part, "conf*": CONF[a], "found": T.loc[m, f"found_{a}"].mean(), "solved": T.loc[m, f"solved_{a}"].mean()})
display(pd.DataFrame(summary).set_index(["arm", "split"]).round(3))

In [ ]:
T["R_miss"] = ~T[[f"solved_{a}" for a in FULL_AVAIL]].any(axis=1)
T["R_notfound"] = ~T[[f"found_{a}" for a in FULL_AVAIL]].any(axis=1)
T["n_solved"] = T[[f"solved_{a}" for a in FULL_AVAIL]].sum(axis=1)
out = []
for part in ("dev", "val"):
    d = T[T.part == part]
    row = {"split": part, "objects": len(d), "R_miss": d.R_miss.mean(), "R_notfound": d.R_notfound.mean()}
    row.update({f"solved by {k} of {len(FULL_AVAIL)}": (d.n_solved == k).mean() for k in range(len(FULL_AVAIL) + 1)})
    out.append(row)
display(pd.DataFrame(out).set_index("split").round(3))
if "D1_s0" in FULL_AVAIL and "D1_s1" in FULL_AVAIL:
    for part in ("dev", "val"):
        d = T[T.part == part]
        print(f"{part}: the two D1 seeds disagree on {(d.solved_D1_s0 != d.solved_D1_s1).mean():.3f} of the objects (solved by exactly one): the seed-level noise of the signal")

## Stage 3 · Populations

`R_miss` and `R_notfound` by attribute with the group size and the **lift** over the base rate, dev next to val. If one population carries the struggle it shows here; if the struggle is spread evenly the lifts stay near 1. Groups under 20 objects are shown but not interpreted.

In [ ]:
ATTRS = ("class", "bucket", "crowded", "border", "bright_q", "scene_q")


def rate_table(signal):
    parts = {}
    for part in ("dev", "val"):
        d = T[T.part == part]
        t = pd.concat([d.groupby(by, observed=True)[signal].agg(["size", "mean"]).set_axis(["n", "rate"], axis=1).rename(index=lambda k, by=by: f"{by}={k}") for by in ATTRS])
        t["lift"] = t.rate / d[signal].mean()
        parts[part] = t
    return pd.concat(parts, axis=1)


for signal in ("R_miss", "R_notfound"):
    print(f"=== {signal}: base rate dev {T[T.part == 'dev'][signal].mean():.3f}, val {T[T.part == 'val'][signal].mean():.3f}")
    display(rate_table(signal).round(3))
    for part in ("dev", "val"):
        print(f"{signal}, class x size bucket, {part}")
        display(T[T.part == part].pivot_table(index="class", columns="bucket", values=signal, aggfunc=["mean", "size"], observed=True).round(3))

## Stage 4 · What the detectors do with the objects they find

For each given class, over the three full-data models pooled (one row per object and model): the share not found, typed as the given class, or typed as another class. Rare types absorbed by Cargo and Box would show here, from the detectors' own predictions.

In [ ]:
def outcome_table(part):
    d = T[T.part == part]
    P = np.concatenate([d[f"pred_{a}"].to_numpy() for a in FULL_AVAIL]); y = np.tile(d.cls.to_numpy(), len(FULL_AVAIL))
    rows = []
    for k in range(len(NAMES)):
        p = P[y == k]
        rows.append({"given class": SHORT[k], "objects": int((d.cls == k).sum()), "not found": (p < 0).mean(), "typed as given": (p == k).mean(),
                     **{f"typed {SHORT[m]}": (p == m).mean() for m in range(len(NAMES)) if m != k}})
    return pd.DataFrame(rows).set_index("given class")


for part in ("dev", "val"):
    print(part)
    display(outcome_table(part).round(3))

## Stage 5 · Prediction, then test

A ridge-regularised logistic regression on the attributes only (class, log size, log crowding, border, log objects in the scene, scene brightness, log sharpness) is **fitted on dev** to predict dev `R_miss`, and scored on **val**, whose outcomes it never saw. AUC and the hit rate in the top fifth of predicted risk (against the base rate) carry image-bootstrap intervals.
Size alone (smaller = riskier) is shown as the yardstick: the attribute model is only worth reporting as more than "small things are missed" if it beats it.

In [ ]:
def design(df, ref=None):
    X = np.column_stack([np.eye(len(NAMES))[df.cls.to_numpy()], np.log(df["size"].to_numpy()), np.log1p(df.crowd.to_numpy()), df.border.to_numpy(float),
                         np.log(df.n_scene.to_numpy()), df.bright.to_numpy(), np.log(df.sharp.to_numpy() + 1e-3)])
    k = len(NAMES)
    mu, sd = ref if ref is not None else (X[:, k:].mean(0), X[:, k:].std(0) + 1e-9)
    X[:, k:] = (X[:, k:] - mu) / sd
    return np.column_stack([np.ones(len(X)), X]), (mu, sd)


def fit_logit(X, y, lam=3.0, iters=30):
    w = np.zeros(X.shape[1]); R = lam * np.eye(X.shape[1]); R[0, 0] = 0
    for _ in range(iters):
        p = 1 / (1 + np.exp(-X @ w))
        w += np.linalg.solve(X.T @ (X * (p * (1 - p))[:, None]) + R, X.T @ (y - p) - R @ w)
    return w


def auc(score, y):
    r = pd.Series(score).rank().to_numpy(); n1 = int(y.sum()); n0 = len(y) - n1
    return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)) if n1 and n0 else float("nan")


def top_fifth(score, y):
    k = max(int(round(0.2 * len(y))), 1)
    return float(y[np.argsort(-score)[:k]].mean())


def boot(images, score, y, reps=BOOTSTRAP, seed=SEED):
    rng = np.random.default_rng(seed)
    groups = pd.Series(np.arange(len(y))).groupby(images.to_numpy()).apply(lambda s: s.to_numpy()).tolist()
    out = []
    for _ in range(reps):
        idx = np.concatenate([groups[k] for k in rng.integers(0, len(groups), len(groups))])
        if 0 < y[idx].sum() < len(idx):
            out.append((auc(score[idx], y[idx]), top_fifth(score[idx], y[idx]) / y[idx].mean()))
    return np.percentile(np.array(out), [2.5, 97.5], axis=0)


devT, valT = T[T.part == "dev"], T[T.part == "val"]
Xd, ref = design(devT)
rows = []
for target in ("R_miss", "R_notfound"):
    w = fit_logit(Xd, devT[target].to_numpy(float))
    for who, score_fn in (("attribute model", lambda df: design(df, ref)[0] @ w), ("size alone", lambda df: -df["size"].to_numpy())):
        for label, df in (("dev (fitted here)", devT), ("val (test)", valT)):
            s, y = score_fn(df), df[target].to_numpy(float)
            lo, hi = boot(df.image, s, y)
            rows.append({"target": target, "predictor": who, "scored on": label, "n": len(df), "base rate": y.mean(), "AUC": auc(s, y), "AUC lo": lo[0], "AUC hi": hi[0],
                         "top-fifth hit rate": top_fifth(s, y), "lift": top_fifth(s, y) / y.mean(), "lift lo": lo[1], "lift hi": hi[1]})
    if target == "R_miss":
        W_MISS = w
RESULT = pd.DataFrame(rows).set_index(["target", "predictor", "scored on"])
display(RESULT.round(3))
row = RESULT.loc[("R_miss", "attribute model", "val (test)")]
verdict = "SUPPORTED" if (row["AUC lo"] > 0.5 and row["lift"] >= 1.3) else "NOT supported"
print("declared rule (val AUC lower bound > 0.5 and top-fifth lift >= 1.3):", verdict + ("" if len(FULL_AVAIL) == len(FULL) else "  [PROVISIONAL: not all three full-data models are available]"))
coef = pd.Series(W_MISS[1:], index=SHORT + ["log size", "log crowd", "border", "log n scene", "brightness", "log sharpness"])
display(coef.round(2).rename("logit coefficient (R_miss, fitted on dev)").to_frame().T)

## Stage 6 · Save

In [ ]:
T.drop(columns=["x1", "y1", "x2", "y2"]).to_csv(OUT_DIR / "instances.csv.gz", index=False)
RESULT.to_csv(OUT_DIR / "predict_then_test.csv")
print(sorted(p.name for p in OUT_DIR.iterdir()))

## Observations

*Results of the Kaggle run: `executed/12_RL_5.2_resist_learning.ipynb` and REPORT §6.2.*